In [3]:
import pandas as pd

# 1. URL to the Parkinson's dataset on the UCI Repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"

# 2. Load the dataset into a Pandas DataFrame
df = pd.read_csv(url)

# 3. Quick look at the data
print("--- First 5 rows of the dataset ---")
print(df.head())

# 4. Preprocessing for your FRQR + ML Pipeline
# The 'name' column is just a recording identifier and must be removed before feature selection.
# The 'status' column is the target variable (1 = Parkinson's, 0 = Healthy).

X = df.drop(columns=['name', 'status'])  # Feature Matrix (22 continuous acoustic variables)
y = df['status']                         # Target Vector

# 5. Display dataset characteristics
print("\n--- Dataset Characteristics ---")
print(f"Total Instances: {df.shape[0]}")
print(f"Total Features (excluding target and ID): {X.shape[1]}")

print("\n--- Class Imbalance ---")
class_counts = y.value_counts()
print(f"Parkinson's Positive (Class 1): {class_counts[1]} instances ({class_counts[1]/len(y)*100:.1f}%)")
print(f"Healthy Negative (Class 0): {class_counts[0]} instances ({class_counts[0]/len(y)*100:.1f}%)")

--- First 5 rows of the dataset ---
             name  MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  \
0  phon_R01_S01_1      119.992       157.302        74.997         0.00784   
1  phon_R01_S01_2      122.400       148.650       113.819         0.00968   
2  phon_R01_S01_3      116.682       131.111       111.555         0.01050   
3  phon_R01_S01_4      116.676       137.871       111.366         0.00997   
4  phon_R01_S01_5      116.014       141.781       110.655         0.01284   

   MDVP:Jitter(Abs)  MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  ...  \
0           0.00007   0.00370   0.00554     0.01109       0.04374  ...   
1           0.00008   0.00465   0.00696     0.01394       0.06134  ...   
2           0.00009   0.00544   0.00781     0.01633       0.05233  ...   
3           0.00009   0.00502   0.00698     0.01505       0.05492  ...   
4           0.00011   0.00655   0.00908     0.01966       0.06425  ...   

   Shimmer:DDA      NHR     HNR  status      RPDE 

In [4]:
import pandas as pd
import numpy as np
import itertools
from sklearn.preprocessing import LabelEncoder

def triangular_mf(x, a, b, c):
    """
    Triangular membership function with foots a, c and peak b (degree 1 at b, 0 at a and c).
    """
    if x <= a or x >= c:
        return 0.0
    elif x <= b:
        if b == a: return 1.0 
        return (x - a) / (b - a)  
    else:
        if c == b: return 1.0
        return (c - x) / (c - b)  

# Load dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"
df = pd.read_csv(url)

# Correctly define the target column
target_col = 'status'

# Drop the 'name' column (it's just a string ID and will break math functions)
if 'name' in df.columns:
    df = df.drop(columns=['name'])

# Separate Features (X) and Target (y)
X = df.drop(columns=[target_col])
y = df[target_col]

# --- ENCODE TEXT TO NUMBERS ---
label_encoders = {}
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype.name == 'category':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

# Define the features list from the cleanly separated X dataframe
features = list(X.columns)

# --- DATA-DRIVEN MEMBERSHIP FUNCTIONS (Using Median + Buffer) ---
membership_funcs = {}
for f in features:
    x_min = df[f].min()
    x_max = df[f].max()
    x_mid = df[f].median() 
    
    std = df[f].std()
    if std == 0: std = 1e-5 
    
    low_mf = (x_min, x_min, x_mid + (0.2 * std))
    med_mf = (x_min, x_mid, x_max)
    high_mf = (x_mid - (0.2 * std), x_max, x_max)
    
    membership_funcs[f] = [low_mf, med_mf, high_mf]

# Fuzzify all features
N = df.shape[0]
mem_vals = {}
for f in features:
    a1, b1, c1 = membership_funcs[f][0] # Low
    a2, b2, c2 = membership_funcs[f][1] # Medium
    a3, b3, c3 = membership_funcs[f][2] # High
    
    mem_array = np.zeros((N, 3))
    
    for i, x in enumerate(df[f]):
        mem_array[i, 0] = triangular_mf(x, a1, b1, c1)
        mem_array[i, 1] = triangular_mf(x, a2, b2, c2)
        mem_array[i, 2] = triangular_mf(x, a3, b3, c3)
        
    mem_vals[f] = mem_array

# Prepare target class memberships
classes = sorted(df[target_col].unique())
if len(classes) != 2:
    raise ValueError("Target column must be binary.")
class0, class1 = classes
mu_class0 = np.array([1.0 if y_val == class0 else 0.0 for y_val in df[target_col]])
mu_class1 = np.array([1.0 if y_val == class1 else 0.0 for y_val in df[target_col]])


def compute_dependency(feature_set):
    """
    Compute fuzzy lower approximations and dependency for a given set of features.
    """
    combos = list(itertools.product([0, 1, 2], repeat=len(feature_set)))

    mem_set = np.zeros((N, len(combos)))
    for k, choices in enumerate(combos):
        for i in range(N):
            values = []
            for idx, f in enumerate(feature_set):
                values.append(mem_vals[f][i, choices[idx]])
            mem_set[i, k] = min(values)

    alphas = np.zeros(len(combos))
    all_min = []
    for k in range(len(combos)):
        M_k = np.maximum(1 - mem_set[:, k], mu_class0)
        alphas[k] = M_k.min()
        Min_k = np.minimum(mem_set[:,k], alphas[k])
        all_min.append(Min_k)

    all_min = np.stack(all_min, axis=1)
    lower_approx_class0 = np.max(all_min, axis=1)

    alphas1 = np.zeros(len(combos))
    all_min1 = []
    for k in range(len(combos)):
        M_k1 = np.maximum(1 - mem_set[:, k], mu_class1)
        alphas1[k] = M_k1.min()
        Min_k1 = np.minimum(mem_set[:, k], alphas1[k])
        all_min1.append(Min_k1)

    all_min1 = np.stack(all_min1, axis=1)
    lower_approx_class1 = np.max(all_min1, axis=1)

    pos_region = np.maximum(lower_approx_class0, lower_approx_class1)
    dependency = np.mean(pos_region)
    
    return dependency

# QuickReduct iterative feature selection
selected = []
current_dep = 0.0
threshold = 0.001  # Define your minimum improvement threshold here

while True:
    best_feature = None
    best_dep = current_dep

    for f in features:
        if f in selected:
            continue
        candidate_set = selected + [f]
        
        dep = compute_dependency(candidate_set)
        
        if dep > best_dep:
            best_dep = dep
            best_feature = f

    # MODIFIED STOPPING CONDITION:
    # Stop if no feature improves dependency OR if the improvement is less than the threshold
    if best_feature is None or (best_dep - current_dep) < threshold:
        # Optional: Print the reason for stopping so you can track it
        improvement = best_dep - current_dep
        print(f"\nStopping: Max dependency increase was {improvement:.4f} (Threshold: {threshold})")
        break

    selected.append(best_feature)
    current_dep = best_dep

print(f"\n======================================")
print(f"FINAL REDUCT: {selected}")
print(f"FINAL DEPENDENCY: {current_dep:.4f}")
print(f"======================================")


Stopping: Max dependency increase was 0.0009 (Threshold: 0.001)

FINAL REDUCT: ['spread1', 'MDVP:Fo(Hz)', 'Shimmer:APQ3', 'spread2', 'MDVP:Flo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:PPQ', 'NHR', 'MDVP:Jitter(%)']
FINAL DEPENDENCY: 0.5092


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, recall_score
from statsmodels.stats.contingency_tables import mcnemar

# 1. Define the sets
reduct_features = ['spread1', 'MDVP:Fo(Hz)', 'Shimmer:APQ3', 'spread2', 
                   'MDVP:Flo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:PPQ', 'NHR', 'MDVP:Jitter(%)']

X_full = X  # All 22 features
X_reduct = X[reduct_features] # Only the 9 FRQR features
y_true = y.values

# 2. Standardization (Crucial for SVM and k-NN)
scaler = StandardScaler()
X_full_scaled = scaler.fit_transform(X_full)
X_reduct_scaled = scaler.fit_transform(X_reduct)

# 3. Define Classifiers
classifiers = {
    "k-NN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree (depth=3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "SVM (Linear, C=1.0)": SVC(kernel='linear', C=1.0, random_state=42)
}

# 4. Setup 10-Fold CV
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print("==========================================================")
print("  FRQR FEATURE REDUCT vs FULL FEATURE SET EVALUATION")
print("==========================================================\n")

for name, clf in classifiers.items():
    print(f"--- Classifier: {name} ---")
    
    # Get cross-validated predictions
    preds_full = cross_val_predict(clf, X_full_scaled, y_true, cv=cv)
    preds_reduct = cross_val_predict(clf, X_reduct_scaled, y_true, cv=cv)
    
    # Calculate Base Metrics
    acc_full = accuracy_score(y_true, preds_full)
    acc_reduct = accuracy_score(y_true, preds_reduct)
    
    recall_full = recall_score(y_true, preds_full)
    recall_reduct = recall_score(y_true, preds_reduct)
    
    print(f"Full Features (22) -> Accuracy: {acc_full:.4f} | Recall: {recall_full:.4f}")
    print(f"FRQR Reduct (9)    -> Accuracy: {acc_reduct:.4f} | Recall: {recall_reduct:.4f}")
    
    # Build Contingency Table for McNemar's Test
    # n11: Both correct
    # n10: Full correct, Reduct wrong
    # n01: Full wrong, Reduct correct
    # n00: Both wrong
    
    n11 = n10 = n01 = n00 = 0
    for i in range(len(y_true)):
        correct_full = (preds_full[i] == y_true[i])
        correct_reduct = (preds_reduct[i] == y_true[i])
        
        if correct_full and correct_reduct:
            n11 += 1
        elif correct_full and not correct_reduct:
            n10 += 1
        elif not correct_full and correct_reduct:
            n01 += 1
        else:
            n00 += 1
            
    # Format table for statsmodels: [[both correct, full correct only], [reduct correct only, both wrong]]
    # Actually, standard McNemar table strictly looks at the disagreements:
    table = [[n11, n10],
             [n01, n00]]
    
    # Perform McNemar's test (exact=True is safer for small datasets)
    result = mcnemar(table, exact=True)
    
    print(f"\nMcNemar's Test Results:")
    print(f"Disagreements -> Full Better: {n10} cases | Reduct Better: {n01} cases")
    print(f"p-value: {result.pvalue:.4f}")
    
    # Interpretation
    alpha = 0.05
    if result.pvalue > alpha:
        print("Conclusion: No significant difference. The 9-feature model is statistically EQUIVALENT to the 22-feature model.")
    else:
        if acc_reduct > acc_full:
            print("Conclusion: Significant difference! The 9-feature model is STATISTICALLY SUPERIOR.")
        else:
            print("Conclusion: Significant difference! The full feature model is statistically superior.")
            
    print("-" * 60 + "\n")

  FRQR FEATURE REDUCT vs FULL FEATURE SET EVALUATION

--- Classifier: k-NN (k=5) ---
Full Features (22) -> Accuracy: 0.9128 | Recall: 0.9388
FRQR Reduct (9)    -> Accuracy: 0.9128 | Recall: 0.9592

McNemar's Test Results:
Disagreements -> Full Better: 8 cases | Reduct Better: 8 cases
p-value: 1.0000
Conclusion: No significant difference. The 9-feature model is statistically EQUIVALENT to the 22-feature model.
------------------------------------------------------------

--- Classifier: Decision Tree (depth=3) ---
Full Features (22) -> Accuracy: 0.8923 | Recall: 0.9320
FRQR Reduct (9)    -> Accuracy: 0.9077 | Recall: 0.9524

McNemar's Test Results:
Disagreements -> Full Better: 6 cases | Reduct Better: 9 cases
p-value: 0.6072
Conclusion: No significant difference. The 9-feature model is statistically EQUIVALENT to the 22-feature model.
------------------------------------------------------------

--- Classifier: SVM (Linear, C=1.0) ---
Full Features (22) -> Accuracy: 0.8718 | Recall: 0.

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score
from statsmodels.stats.contingency_tables import mcnemar

# 1. Define the sets
reduct_features = ['spread1', 'MDVP:Fo(Hz)', 'Shimmer:APQ3', 'spread2', 
                   'MDVP:Flo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:PPQ', 'NHR', 'MDVP:Jitter(%)']

X_full = X  # All 22 features
X_reduct = X[reduct_features] # Only the 9 FRQR features
y_true = y.values

# 2. Perform 80/20 Train-Test Split (Stratified to maintain class imbalance)
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y_true, test_size=0.20, stratify=y_true, random_state=42
)

X_train_reduct, X_test_reduct, _, _ = train_test_split(
    X_reduct, y_true, test_size=0.20, stratify=y_true, random_state=42
)

# 3. Standardization (Fit on train, transform on test to prevent data leakage)
scaler_full = StandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full)
X_test_full_scaled = scaler_full.transform(X_test_full)

scaler_reduct = StandardScaler()
X_train_reduct_scaled = scaler_reduct.fit_transform(X_train_reduct)
X_test_reduct_scaled = scaler_reduct.transform(X_test_reduct)

# 4. Define Classifiers
classifiers = {
    "k-NN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree (depth=3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "SVM (Linear, C=1.0)": SVC(kernel='linear', C=1.0, random_state=42)
}

print("==========================================================")
print("  FRQR FEATURE REDUCT vs FULL FEATURE SET (80/20 SPLIT)")
print("==========================================================\n")

for name, clf in classifiers.items():
    print(f"--- Classifier: {name} ---")
    
    # Train and predict for FULL feature set
    clf.fit(X_train_full_scaled, y_train)
    preds_full = clf.predict(X_test_full_scaled)
    
    # Train and predict for REDUCT feature set
    clf.fit(X_train_reduct_scaled, y_train)
    preds_reduct = clf.predict(X_test_reduct_scaled)
    
    # Calculate Base Metrics on Test Set (20% of data)
    acc_full = accuracy_score(y_test, preds_full)
    acc_reduct = accuracy_score(y_test, preds_reduct)
    
    recall_full = recall_score(y_test, preds_full)
    recall_reduct = recall_score(y_test, preds_reduct)
    
    print(f"Full Features (22) -> Accuracy: {acc_full:.4f} | Recall: {recall_full:.4f}")
    print(f"FRQR Reduct (9)    -> Accuracy: {acc_reduct:.4f} | Recall: {recall_reduct:.4f}")
    
    # Build Contingency Table for McNemar's Test on the Test Set
    n11 = n10 = n01 = n00 = 0
    for i in range(len(y_test)):
        correct_full = (preds_full[i] == y_test[i])
        correct_reduct = (preds_reduct[i] == y_test[i])
        
        if correct_full and correct_reduct:
            n11 += 1
        elif correct_full and not correct_reduct:
            n10 += 1
        elif not correct_full and correct_reduct:
            n01 += 1
        else:
            n00 += 1
            
    # Format table for statsmodels
    table = [[n11, n10],
             [n01, n00]]
    
    # Perform McNemar's test (exact=True for smaller datasets)
    result = mcnemar(table, exact=True)
    
    print(f"\nMcNemar's Test Results (Test Set N={len(y_test)}):")
    print(f"Disagreements -> Full Better: {n10} cases | Reduct Better: {n01} cases")
    print(f"p-value: {result.pvalue:.4f}")
    
    # Interpretation
    alpha = 0.05
    if result.pvalue > alpha:
        print("Conclusion: No significant difference. The 9-feature model is statistically EQUIVALENT to the 22-feature model.")
    else:
        if acc_reduct > acc_full:
            print("Conclusion: Significant difference! The 9-feature model is STATISTICALLY SUPERIOR.")
        else:
            print("Conclusion: Significant difference! The full feature model is statistically superior.")
            
    print("-" * 60 + "\n")

  FRQR FEATURE REDUCT vs FULL FEATURE SET (80/20 SPLIT)

--- Classifier: k-NN (k=5) ---
Full Features (22) -> Accuracy: 0.9231 | Recall: 0.9655
FRQR Reduct (9)    -> Accuracy: 0.8974 | Recall: 0.8966

McNemar's Test Results (Test Set N=39):
Disagreements -> Full Better: 3 cases | Reduct Better: 2 cases
p-value: 1.0000
Conclusion: No significant difference. The 9-feature model is statistically EQUIVALENT to the 22-feature model.
------------------------------------------------------------

--- Classifier: Decision Tree (depth=3) ---
Full Features (22) -> Accuracy: 0.7179 | Recall: 0.7586
FRQR Reduct (9)    -> Accuracy: 0.8974 | Recall: 0.8621

McNemar's Test Results (Test Set N=39):
Disagreements -> Full Better: 0 cases | Reduct Better: 7 cases
p-value: 0.0156
Conclusion: Significant difference! The 9-feature model is STATISTICALLY SUPERIOR.
------------------------------------------------------------

--- Classifier: SVM (Linear, C=1.0) ---
Full Features (22) -> Accuracy: 0.9487 | Reca